# American Option Pricing via Longstaff-Schwartz

## Why Feynman-Kac Doesn't Directly Apply

European options have a fixed exercise date, so the price is a simple expectation:

$$V_{\text{Euro}} = e^{-rT}\,\mathbb{E}[\text{payoff}(S_T)]$$

American options can be exercised **at any time** $\tau \in [0, T]$. The price becomes an **optimal stopping problem**:

$$V_{\text{Amer}} = \sup_{\tau \in [0, T]} e^{-r\tau}\,\mathbb{E}[\text{payoff}(S_\tau)]$$

The holder must decide at each time step: **exercise now** (receive the intrinsic value) or **continue** (hope for a higher payoff later). This decision depends on the **continuation value** — the expected discounted future payoff if we don't exercise.

## The Longstaff-Schwartz Algorithm (LSM)

The key idea of Longstaff and Schwartz (2001) is to **estimate the continuation value using regression**:

1. Simulate $N$ asset price paths on a time grid $t_0, t_1, \ldots, t_M = T$
2. At expiry $t_M$: exercise value = payoff($S_T$)
3. **Work backward** from $t_{M-1}$ to $t_1$:
   - For paths that are in-the-money (ITM), regress the **discounted future cashflows** on polynomial basis functions of $S_{t_k}$
   - The fitted regression gives the **estimated continuation value** $\hat{C}(S_{t_k})$
   - If payoff($S_{t_k}$) $> \hat{C}(S_{t_k})$: exercise early (replace future cashflow with current payoff)
   - Otherwise: continue holding
4. Final price = average of discounted optimal cashflows

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, log, sqrt, exp
%matplotlib inline

# Black-Scholes analytical formulas (European) for comparison
def bs_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def bs_put(S, K, r, T, sigma):
    d1 = (log(S/K) + (r + sigma**2/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return -S * bs_cdf(-d1) + K * exp(-r*T) * bs_cdf(-d2)

def bs_call(S, K, r, T, sigma):
    d1 = (log(S/K) + (r + sigma**2/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S * bs_cdf(d1) - K * exp(-r*T) * bs_cdf(d2)

## Step 1: Simulate GBM Paths

First, we simulate full asset price paths (not just terminal values like in Chapter 4). We need the entire trajectory because American options can be exercised at any time step.

In [ ]:
def simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths):
    """
    Simulate GBM paths under the risk-neutral measure.
    
    Returns: array of shape (n_steps+1, n_paths), where row 0 is S0.
    """
    dt = T / n_steps
    Z = np.random.randn(n_steps, n_paths)
    log_increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_S = np.log(S0) + np.cumsum(log_increments, axis=0)
    # Prepend S0
    S = np.exp(np.vstack([np.full((1, n_paths), np.log(S0)), log_S]))
    return S

# Test: simulate and plot a few paths
np.random.seed(42)
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2
n_steps, n_paths = 50, 100000

S = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths)
t_grid = np.linspace(0, T, n_steps + 1)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.plot(t_grid, S[:, i], alpha=0.7, linewidth=0.8)
plt.axhline(K, color='k', linestyle='--', alpha=0.5, label=f'K = {K}')
plt.xlabel('Time')
plt.ylabel('$S_t$')
plt.title(f'Sample GBM Paths ({n_steps} steps)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f"Simulated {n_paths} paths with {n_steps} time steps")

## Step 2: The Longstaff-Schwartz Algorithm

The core algorithm works backward from expiry. At each time step $t_k$:

1. Identify **in-the-money (ITM)** paths — only these have a meaningful exercise decision
2. For ITM paths, compute the **discounted future cashflow** $Y$ (what you'd get if you continued)
3. **Regress** $Y$ on polynomial basis functions of $S_{t_k}$ to get $\hat{C}(S_{t_k})$ (estimated continuation value)
4. **Exercise** if intrinsic value $>$ $\hat{C}$; otherwise continue

We use Laguerre polynomials (weighted) as the basis, following the original paper. In practice, simple powers $\{1, S, S^2\}$ also work well.

In [ ]:
def lsm_american_option(S, K, r, T, payoff_func, poly_degree=3):
    """
    Longstaff-Schwartz algorithm for American option pricing.
    
    S: asset price paths, shape (n_steps+1, n_paths)
    K: strike price
    r: risk-free rate
    T: time to expiration
    payoff_func: function(S, K) -> intrinsic value (vectorized)
    poly_degree: degree of polynomial basis for regression
    
    Returns: (price, std_error, exercise_times)
    """
    n_steps = S.shape[0] - 1
    n_paths = S.shape[1]
    dt = T / n_steps
    
    # Cashflow matrix: when and how much each path receives
    # cashflow[i] = (discounted) payoff for path i at its optimal exercise time
    cashflow = payoff_func(S[-1], K)  # initialize with terminal payoff
    exercise_time = np.full(n_paths, n_steps)  # track when each path exercises
    
    # Backward induction from t_{M-1} to t_1
    for k in range(n_steps - 1, 0, -1):
        intrinsic = payoff_func(S[k], K)
        itm = intrinsic > 0  # in-the-money paths
        
        if np.sum(itm) == 0:
            continue
        
        # Discounted future cashflow for ITM paths
        # Discount from the exercise time back to step k
        steps_ahead = exercise_time[itm] - k
        Y = cashflow[itm] * np.exp(-r * dt * steps_ahead)
        
        # Regression: fit continuation value as polynomial of S[k]
        X = S[k, itm]
        # Normalize X for numerical stability
        X_norm = X / K
        basis = np.column_stack([X_norm**p for p in range(poly_degree + 1)])
        
        # Least squares regression
        coeffs, _, _, _ = np.linalg.lstsq(basis, Y, rcond=None)
        continuation_value = basis @ coeffs
        
        # Exercise decision: exercise if intrinsic > continuation
        exercise = intrinsic[itm] > continuation_value
        
        # Update cashflow and exercise time for paths that exercise early
        itm_indices = np.where(itm)[0]
        exercise_indices = itm_indices[exercise]
        cashflow[exercise_indices] = intrinsic[itm][exercise]
        exercise_time[exercise_indices] = k
    
    # Discount all cashflows back to t=0
    discounted = cashflow * np.exp(-r * dt * exercise_time)
    
    price = np.mean(discounted)
    std_err = np.std(discounted) / np.sqrt(n_paths)
    
    return price, std_err, exercise_time

## Step 3: Price American Put and Compare with European

For an American **call** on a non-dividend-paying stock, early exercise is never optimal (the call is worth more alive than dead), so $V_{\text{Amer}} = V_{\text{Euro}}$.

For an American **put**, early exercise can be optimal when the stock drops well below the strike — it may be better to receive $K - S$ now and invest at rate $r$ than to wait. So $V_{\text{Amer}} \geq V_{\text{Euro}}$ for puts, and the difference is the **early exercise premium**.

In [ ]:
put_payoff = lambda S, K: np.maximum(K - S, 0)
call_payoff = lambda S, K: np.maximum(S - K, 0)

# European prices (analytical)
euro_put = bs_put(S0, K, r, T, sigma)
euro_call = bs_call(S0, K, r, T, sigma)

# American prices (LSM)
amer_put, amer_put_err, put_ex_times = lsm_american_option(S, K, r, T, put_payoff)
amer_call, amer_call_err, call_ex_times = lsm_american_option(S, K, r, T, call_payoff)

print("=== American vs European Option Prices ===")
print(f"{'':20s} {'European (BS)':>14s} {'American (LSM)':>16s} {'Std Error':>10s} {'Premium':>10s}")
print(f"{'Put':20s} {euro_put:14.6f} {amer_put:16.6f} {amer_put_err:10.6f} {amer_put - euro_put:10.6f}")
print(f"{'Call':20s} {euro_call:14.6f} {amer_call:16.6f} {amer_call_err:10.6f} {amer_call - euro_call:10.6f}")
print()
print("Note: American call premium should be ~0 (no early exercise for non-dividend stocks)")
print(f"      American put premium should be > 0 (early exercise is sometimes optimal)")

## Step 4: Visualize the Early Exercise Behavior

When does early exercise happen? Let's look at the distribution of exercise times for the American put.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Exercise time distribution for American put
ex_t_put = put_ex_times * (T / n_steps)
early_put = put_ex_times < n_steps
axes[0].hist(ex_t_put[early_put], bins=n_steps, alpha=0.7, density=True, label='Early exercise')
axes[0].axvline(T, color='r', linestyle='--', label=f'Expiry T={T}')
axes[0].set_xlabel('Exercise time')
axes[0].set_ylabel('Density')
axes[0].set_title(f'American Put Exercise Times\n({np.mean(early_put)*100:.1f}% exercise early)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Exercise time distribution for American call
ex_t_call = call_ex_times * (T / n_steps)
early_call = call_ex_times < n_steps
axes[1].hist(ex_t_call, bins=n_steps, alpha=0.7, density=True, label='At expiry')
axes[1].axvline(T, color='r', linestyle='--', label=f'Expiry T={T}')
axes[1].set_xlabel('Exercise time')
axes[1].set_ylabel('Density')
axes[1].set_title(f'American Call Exercise Times\n({np.mean(early_call)*100:.1f}% exercise early)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Early Exercise Boundary

The **exercise boundary** $S^*(t)$ separates the "exercise" region from the "continue" region. For a put, you exercise when $S_t < S^*(t)$. Let's estimate this boundary from our simulation.

In [ ]:
# Estimate exercise boundary: for each time step, find the max S at which exercise occurred
exercise_boundary = np.full(n_steps + 1, np.nan)
for k in range(1, n_steps + 1):
    exercised_at_k = (put_ex_times == k)
    if np.any(exercised_at_k):
        exercise_boundary[k] = np.max(S[k, exercised_at_k])

fig, ax = plt.subplots(figsize=(10, 5))

# Plot a few paths that exercised early
early_paths = np.where(put_ex_times < n_steps)[0][:8]
for i in early_paths:
    k = put_ex_times[i]
    ax.plot(t_grid[:k+1], S[:k+1, i], alpha=0.5, linewidth=0.8)
    ax.scatter(t_grid[k], S[k, i], color='red', s=30, zorder=5)

# Plot exercise boundary
valid = ~np.isnan(exercise_boundary)
ax.plot(t_grid[valid], exercise_boundary[valid], 'r-', linewidth=2, label='Exercise boundary $S^*(t)$')
ax.axhline(K, color='k', linestyle='--', alpha=0.3, label=f'Strike K={K}')
ax.fill_between(t_grid[valid], 0, exercise_boundary[valid], alpha=0.1, color='red', label='Exercise region')
ax.set_xlabel('Time')
ax.set_ylabel('$S_t$')
ax.set_title('American Put: Early Exercise Boundary and Sample Paths')
ax.set_ylim(40, 120)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: Early Exercise Premium vs Moneyness

How does the early exercise premium vary with the stock price? Deep ITM puts should have a larger premium because early exercise is more attractive.

In [ ]:
S0_range = np.linspace(70, 130, 15)
euro_prices = []
amer_prices = []
amer_errors = []

for s0 in S0_range:
    S_tmp = simulate_gbm_paths(s0, r, sigma, T, n_steps, 50000)
    ep = bs_put(s0, K, r, T, sigma)
    ap, ae, _ = lsm_american_option(S_tmp, K, r, T, put_payoff)
    euro_prices.append(ep)
    amer_prices.append(ap)
    amer_errors.append(ae)

euro_prices = np.array(euro_prices)
amer_prices = np.array(amer_prices)
amer_errors = np.array(amer_errors)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Price comparison
ax1.plot(S0_range, euro_prices, 'b--', linewidth=2, label='European put (BS)')
ax1.plot(S0_range, amer_prices, 'r-', linewidth=2, label='American put (LSM)')
ax1.fill_between(S0_range, amer_prices - 2*amer_errors, amer_prices + 2*amer_errors, 
                 alpha=0.2, color='r')
ax1.plot(S0_range, np.maximum(K - S0_range, 0), 'k:', alpha=0.5, label='Intrinsic value')
ax1.set_xlabel('$S_0$')
ax1.set_ylabel('Put price')
ax1.set_title('European vs American Put')
ax1.legend()
ax1.grid(alpha=0.3)

# Early exercise premium
premium = amer_prices - euro_prices
ax2.bar(S0_range, premium, width=3.5, alpha=0.7)
ax2.set_xlabel('$S_0$')
ax2.set_ylabel('Premium')
ax2.set_title('Early Exercise Premium (American - European)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

| Property | European Option | American Option |
|----------|:--------------:|:--------------:|
| Exercise | At expiry only | Any time up to expiry |
| PDE | Black-Scholes PDE | Variational inequality (free boundary) |
| Stochastic form | $e^{-rT}\,\mathbb{E}[\text{payoff}(S_T)]$ | $\sup_\tau\, e^{-r\tau}\,\mathbb{E}[\text{payoff}(S_\tau)]$ |
| MC method | Direct Feynman-Kac | Longstaff-Schwartz (regression) |
| Call premium | N/A (same price) | 0 for non-dividend stocks |
| Put premium | N/A | > 0 (early exercise optimal for deep ITM) |

Key observations:
- The **early exercise boundary** $S^*(t)$ increases toward $K$ as $t \to T$ — near expiry, only very deep ITM puts are exercised early.
- The **premium is largest for ITM puts** ($S_0 \ll K$) where the time value of money from receiving $K - S$ early outweighs the optionality of waiting.
- American call = European call for non-dividend stocks — there's no incentive to exercise early because you'd forfeit the remaining time value.